In [11]:
import os
from pathlib import Path
from PIL import Image
import random
import shutil

In [ ]:
IMAGES_DIR=Path("ExDark")
ANNO_DIR=Path("ExDark_Annno")
OUTPUT_DIR=Path("yolo_dataset")

In [ ]:
# ExDark dataset have 12 classes which need to mapped into integers for YOLO training
CLASSES = {
    "Bicycle": 0, "Boat": 1, "Bottle": 2, "Bus": 3, 
    "Car": 4, "Cat": 5, "Chair": 6, "Cup": 7, 
    "Dog": 8, "Motorbike": 9, "People": 10, "Table": 11
}

SPLIT_RATIO = 0.8 # for train test split

In [ ]:
#it creates required yolo directory as we get raw images and YOLO folows some directory format
def create_yolo_dir(base_path:Path):
    for split in ["train","val"]:
        (base_path / "images" / split).mkdir(parents=True,exist_ok=True)
        (base_path / "labels" / split).mkdir(parents=True,exist_ok=True)

In [ ]:
#we get annotation of bounding boxes and classes in .txt but they are not in required format to train YOLO Models
def process_annotation(txt_path : Path,img_width:int,img_height:int)->list[str]:
    """Parses the ExDark .txt file and converts to YOLO normalized format."""
    yolo_lines=[]

    with open(txt_path,"r") as file:
        
        for line in file:
            line=line.strip()

            if line.startswith("%") or not line:
                continue
                
            parts=line.split()

            if len(parts) < 5:
                continue
                
            class_name=parts[0]

            if class_name not in CLASSES:
                continue

            class_id=CLASSES[class_name]

            l,t,w,h=map(float,parts[1:5])

            x_center = (l + (w/2.0)) / img_width
            y_center= (t + (h/2.0)) / img_height
            w_norm=w/img_width
            h_norm=h/img_height

            # Format: <class_index> <x_center> <y_center> <width> <height>
            yolo_lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}\n")

        return yolo_lines

In [ ]:
def main():
    print("Starting dataset conversion...")
    create_yolo_dir(OUTPUT_DIR)

    for class_folder in IMAGES_DIR.iterdir():
        if not class_folder.is_dir():
            continue

        print(f"Processing folder {class_folder.name}...")

        images=list(class_folder.glob("*.jpg")) + list(class_folder.glob("*.png")) + list(class_folder.glob("*.JPEG")) + list(class_folder.glob("*.JPG"))
        random.shuffle(images)

        split_index=int(len(images)*SPLIT_RATIO)
        train_images=images[:split_index]
        val_images=images[split_index:]

        for split_name,img_list in [("train",train_images),("val",val_images)]:
            for img_path in img_list:
                
                # Match image with its annotation file
                anno_filename=img_path.name + ".txt"
                txt_path = ANNO_DIR / class_folder.name / anno_filename

                if not txt_path.exists():
                    txt_path=ANNO_DIR / class_folder.name / (img_path.stem + ".txt")
                
                if not txt_path.exists():
                    continue
            
                
                try:
                    with Image.open(img_path) as img:
                        img_width,img_height=img.size
                    
                except Exception as e:
                    print(f"Error reading image {img_path} : {e}")
                    continue
            
                yolo_annotations=process_annotation(txt_path,img_width,img_height)

                if yolo_annotations:
                    # copy image to YOLO structure
                    dest_img_path=OUTPUT_DIR / "images" / split_name / img_path.name
                    shutil.copy(img_path,dest_img_path)

                    dest_txt_path=OUTPUT_DIR / "labels" / split_name / (img_path.stem + ".txt")
                    with open(dest_txt_path,"w") as out_file:
                        out_file.writelines(yolo_annotations)


    print("\nDataset conversion complete")

if __name__=="__main__":
    main()

Initializing Project Night Owl dataset conversion...
Processing folder Cat...
Processing folder Car...
Processing folder Cup...
Processing folder Dog...
Processing folder Boat...
Processing folder Chair...
Processing folder Bus...
Processing folder Motorbike...
Processing folder Table...
Processing folder People...
Processing folder Bicycle...
Processing folder Bottle...

Dataset conversion complete! Ready for YOLOv8.
